In [ ]:
from src.compiler.qccd_ion_routing import *
from src.simulator.qccd_circuit import *
d = 4
# safe to have either barriers or go back
barrierThreshold = np.inf
goBackThreshold = 0
for trapCapacity in [2]:

    circuit = QCCDCircuit.generated(
        "surface_code:rotated_memory_z",
        rounds=1,
        distance=d,
    )
    nqubitsNeeded = 2 * d**2 - 1

    nrowsNeeded = int(np.sqrt(nqubitsNeeded))+2

    # wiseArch = QCCDWiseArch(m=int(np.sqrt(trapCapacity*nqubitsNeeded/2))+1, n=int(np.sqrt(2*nqubitsNeeded/trapCapacity))+1, k=trapCapacity)
    wiseArch =QCCDWiseArch(m=6, n=6, k=trapCapacity)
    arch, (instructions, opBarriers) = circuit.processCircuitWiseArch(wiseArch=wiseArch)
    
    arch.refreshGraph()

    fig,ax =plt.subplots()
    arch.display(fig, ax, title='map complete', showLabels=False)
    # fig.set_size_inches(arch.WINDOW_SIZE[0]*1.5, arch.WINDOW_SIZE[1]*1.5)

    opBarriers = opBarriers if trapCapacity<=barrierThreshold else []
    allOps, barriers = ionRoutingWISEArch(arch, wiseArch, instructions)
    parallelOpsMap = paralleliseOperationsWithBarriers(allOps, barriers)
    parallelOps = list(dict(parallelOpsMap).values())

    errs = circuit.simulate(allOps, isWISEArch=True)

    arch = circuit.resetArch()
    arch.refreshGraph()

    trapSet = set()
    junctionSet = set()
    for op in allOps:
        for c in op.involvedComponents:
            if isinstance(c, Trap):
                trapSet.add(c)
            elif isinstance(c, Junction):
                junctionSet.add(c)


    Njz = len(junctionSet) # each junction is one zone
    Nlz = len(trapSet)*trapCapacity # each trap is k zones
    Nde_lz = 10
    Nde_jz = 20
    Nse_z = 10
   

    Njz = int(np.ceil(nqubitsNeeded / (2*(trapCapacity-1))) )# 2 traps per junction
    Nlz = nqubitsNeeded-Njz
    Nde = Nde_lz*Nlz+Nde_jz*Njz
    Nse = Nse_z*(Njz+Nlz)

    Num_electrodes = Nde+Nse
    Num_DACs = Num_electrodes


    print(f"total number of qubit operations: {len(instructions)}")
    print(f"total number of operations: {len(allOps)}")
    print(f"time for operations: {max(parallelOpsMap.keys())}")
    print(f'Number of Linear Zones: {Nlz}')
    print(f"Number of Junction Zones: {Njz}")
    print(f"Number of Electrodes: {Num_electrodes}={Nse}+{Nde}")
    print(f"Errors: {errs}")


In [ ]:
from src.compiler.qccd_ion_routing import *
from src.simulator.qccd_circuit import *
d = 4
# safe to have either barriers or go back
barrierThreshold = np.inf
goBackThreshold = 0
for trapCapacity in [2]:

    circuit = QCCDCircuit.generated(
        "surface_code:rotated_memory_z",
        rounds=1,
        distance=d,
    )
    nqubitsNeeded = 2 * d**2 - 1

    nrowsNeeded = int(np.sqrt(nqubitsNeeded))+2

    wiseArch = QCCDWiseArch(m=int(np.sqrt(trapCapacity*nqubitsNeeded/2))+1, n=int(np.sqrt(2*nqubitsNeeded/trapCapacity))+1, k=trapCapacity)
    arch, (instructions, opBarriers) = circuit.processCircuitWiseArch(wiseArch=wiseArch)
    
    print(instructions)
    oldPositions = {}
    for idx, (ion, pos) in circuit._ionMapping.items():
        oldPositions[idx] = ion.pos
        ion.set(ion.idx, pos[0], pos[1], parent=ion.parent)

    fig,ax =plt.subplots()
    arch1 = arch
    arch1.refreshGraph()
    edgesDups = []
    edgesPos = []
    ionsInvolved = set()
    score = 0
    for op in instructions:
        if not isinstance(op, TwoQubitMSGate):
            continue
        if not ionsInvolved.isdisjoint(op.ions):
            score+=1
            ionsInvolved=set()
        edgesDups.append((op.ions, score))
        edgesPos.append((op.ionsActedIdxs, score))
        ionsInvolved=ionsInvolved.union(op.ions)
    scores = [score for ((ion1,ion2), score) in edgesDups]
    color_from_score = {s: (5+i*i)*0 for i, s in enumerate( sorted(list(set(scores)), reverse=True))}
    arch1._manipulationTraps.append(([(ion1.idx, ion2.idx) for ((ion1,ion2), score) in edgesDups], [color_from_score[score] for ((ion1,ion2), score) in edgesDups]))
    arch1.display(fig, ax, showLabels=False, showEdges=False, show_junction=False)
    fig.set_size_inches(arch.WINDOW_SIZE[0]*0.9, arch.WINDOW_SIZE[1]*0.9)
    arch1._manipulationTraps = arch1._manipulationTraps[:-1]
    for idx, (ion, pos) in circuit._ionMapping.items():
        ion.set(ion.idx, oldPositions[idx][0], oldPositions[idx][1], parent=ion.parent)

    arch.refreshGraph()

    fig,ax =plt.subplots()
    arch.display(fig, ax, title='map complete', showLabels=False)
    fig.set_size_inches(arch.WINDOW_SIZE[0]*1.5, arch.WINDOW_SIZE[1]*1.5)

    opBarriers = opBarriers if trapCapacity<=barrierThreshold else []
    allOps, barriers = ionRouting(arch, instructions, trapCapacity)
    parallelOpsMap = paralleliseOperationsWithBarriers(allOps, barriers, isWiseArch=True)
    parallelOps = list(dict(parallelOpsMap).values())

    errs = circuit.simulate(allOps, isWISEArch=True)

    arch = circuit.resetArch()
    arch.refreshGraph()

    trapSet = set()
    junctionSet = set()
    for op in allOps:
        for c in op.involvedComponents:
            if isinstance(c, Trap):
                trapSet.add(c)
            elif isinstance(c, Junction):
                junctionSet.add(c)


    Njz = len(junctionSet) # each junction is one zone
    Nlz = len(trapSet)*trapCapacity # each trap is k zones
    Nde_lz = 10
    Nde_jz = 20
    Nse_z = 10
   

    Njz = int(np.ceil(nqubitsNeeded / (2*(trapCapacity-1))) )# 2 traps per junction
    Nlz = nqubitsNeeded-Njz
    Nde = Nde_lz*Nlz+Nde_jz*Njz
    Nse = Nse_z*(Njz+Nlz)

    Num_electrodes = Nde+Nse
    Num_DACs = Num_electrodes


    print(f"total number of qubit operations: {len(instructions)}")
    print(f"total number of operations: {len(allOps)}")
    print(f"time for operations: {max(parallelOpsMap.keys())}")
    print(f'Number of Linear Zones: {Nlz}')
    print(f"Number of Junction Zones: {Njz}")
    print(f"Number of Electrodes: {Num_electrodes}={Nse}+{Nde}")
    print(f"Errors: {errs}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
%matplotlib inline
from IPython.display import HTML, clear_output, display
import time
arch = circuit.resetArch()
arch.refreshGraph()
fig,ax=plt.subplots()
parallelOpsTimes = sorted(parallelOpsMap.keys())


def update(frame, _arch):
    ax.clear()
    # Clear the output for dynamic display in Jupyter notebook
    clear_output(wait=True)
    if frame>0:
        op: ParallelOperation = parallelOps[frame-1]
        title = f"Operation {round(parallelOpsTimes[frame-1],6)}: {op.label}"
        _arch.display(fig, ax, title, operation=op, runOps=True, showLabels=False)
    else:
        _arch.display(fig, ax, showLabels=False)
    # time.sleep(3)
    return display(fig)
import time
time.sleep(10)
ani = FuncAnimation(fig, lambda frame: update(frame, arch), frames=len(parallelOps)+1, repeat=False)
HTML(ani.to_jshtml())

In [ ]:
!python -m pip install python-sat